In [ ]:
import csv
import json
import random
import time
import datetime
import uuid

def generate_random_event_id():
    """生成随机事件ID（5-10位数字）"""
    return str(random.randint(10000, 9999999999))

def generate_random_user_id():
    """生成随机用户ID（guest-N或数字ID）"""
    if random.random() < 0.5:
        return f"guest-{random.randint(1, 10000)}"
    else:
        return str(random.randint(1, 1000000))

def generate_random_timestamp():
    """生成随机时间戳（ISO8601或Unix）"""
    if random.random() < 0.7:
        # 70%概率生成ISO8601格式
        start = datetime.datetime(2023, 1, 1)
        end = datetime.datetime(2023, 12, 31)
        random_time = start + (end - start) * random.random()
        return random_time.isoformat()
    else:
        # 30%概率生成Unix时间戳
        return str(random.randint(1609459200, 1704067200))  # 2021-01-01 到 2024-01-01

def generate_random_event_type():
    """生成随机事件类型"""
    event_types = ["click", "view", "purchase", "add_to_cart"]
    return random.choice(event_types)

def generate_random_device_info():
    """生成随机设备信息（JSON格式）"""
    device_types = ["mobile", "desktop", "tablet"]
    os_list = ["iOS", "Android", "Windows", "macOS"]
    browsers = ["Chrome", "Safari", "Firefox", "Edge"]
    resolutions = ["1920x1080", "1366x768", "1080x1920", "1440x900"]
    
    device_info = {
        "device_type": random.choice(device_types),
        "os": random.choice(os_list),
        "browser": random.choice(browsers),
        "screen_resolution": random.choice(resolutions),
        "app_version": f"{random.randint(1, 3)}.{random.randint(0, 9)}.{random.randint(0, 99)}"
    }
    return json.dumps(device_info)

def generate_random_metadata():
    """生成随机附加文本（包含特殊格式）"""
    metadata_types = [
        f"http://example.com/page_{random.randint(1, 1000)}?param={random.randint(1, 100)}",
        json.dumps({"source": random.choice(["referral", "direct", "social", "search"])}),
        f"{datetime.date(2023, random.randint(1, 12), random.randint(1, 28))} to {datetime.date(2023, random.randint(1, 12), random.randint(1, 28))}",
        f"campaign_id_{random.randint(1000, 9999)}",
        f"user_segment_{random.choice(['A', 'B', 'C', 'D'])}"
    ]
    return random.choice(metadata_types)

def main():
    start_time = time.time()
    rows_to_generate = 1000000
    filename = "large_data.csv"
    
    # 创建事件ID集合确保唯一性
    event_ids = set()
    
    with open(filename, 'w', newline='', encoding='utf-8') as csvfile:
        fieldnames = ['event_id', 'user_id', 'action_time', 'event_type', 'device_info', 'metadata']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        
        # 写入表头
        writer.writeheader()
        
        # 生成数据
        for i in range(rows_to_generate):
            # 确保event_id唯一
            while True:
                event_id = generate_random_event_id()
                if event_id not in event_ids:
                    event_ids.add(event_id)
                    break
            
            # 生成一行数据
            row = {
                'event_id': event_id,
                'user_id': generate_random_user_id(),
                'action_time': generate_random_timestamp(),
                'event_type': generate_random_event_type(),
                'device_info': generate_random_device_info(),
                'metadata': generate_random_metadata()
            }
            
            # 写入行
            writer.writerow(row)
            
            # 打印进度
            if (i + 1) % 100000 == 0:
                print(f"已生成 {i + 1} 行数据...")
    
    end_time = time.time()
    print(f"\n数据生成完成！")
    print(f"文件名: {filename}")
    print(f"总行数: {rows_to_generate}")
    print(f"生成时间: {end_time - start_time:.2f} 秒")

if __name__ == "__main__":
    main()

上述为带脏数据的模拟日志，规模为1000000个。

In [ ]:

import time
import pandas as pd
import polars as pl

# 1. 传统基准: Pandas（读取CSV并统计耗时）
start = time.time()
df_pd = pd.read_csv("large_data.csv")
print(f"Pandas 耗时: {time.time() - start:.3f} 秒")

# 2. 现代多线程: Polars（读取CSV并统计耗时）
start = time.time()
df_pl = pl.read_csv("large_data.csv")
print(f"Polars 耗时: {time.time() - start:.3f} 秒")

# 可选：验证数据读取是否完整（避免空文件/路径错误）
print(f"\n数据总行数 - Pandas: {len(df_pd)}, Polars: {len(df_pl)}")

Pandas 耗时: 2.670 秒
Polars 耗时: 0.285 秒

数据总行数 - Pandas: 1000000, Polars: 1000000


In [1]:
import pandas as pd
import json

def main():
    print("正在读取数据...")
    # 读取数据
    df = pd.read_csv('large_data.csv')
    print(f"数据读取完成，共 {len(df)} 条记录")
    
    # 任务1：event_type字段统计
    print("\n=== 任务1：event_type字段统计 ===")
    event_type_counts = df['event_type'].value_counts().sort_values(ascending=False)
    print(event_type_counts)
    
    # 任务2：user_id查询
    print("\n=== 任务2：user_id查询 ===")
    user_id_minus1_count = df[df['user_id'] == -1].shape[0]
    user_id_guest_count = df[df['user_id'].str.startswith('guest-', na=False)].shape[0]
    
    print(f"user_id为-1的记录数：{user_id_minus1_count}")
    print(f"user_id以guest-开头的记录数：{user_id_guest_count}")
    
    # 任务3：device_info解析
    print("\n=== 任务3：device_info解析（操作系统分布） ===")
    
    # 定义函数解析JSON字符串
    def extract_os(device_info_str):
        try:
            device_info = json.loads(device_info_str)
            return device_info.get('os', 'Unknown')
        except json.JSONDecodeError:
            return 'Error'
    
    # 提取os信息
    df['os'] = df['device_info'].apply(extract_os)
    
    # 统计不同操作系统的设备数
    os_counts = df['os'].value_counts().sort_values(ascending=False)
    print(os_counts)
    
    print("\n分析完成！")

if __name__ == "__main__":
    main()

正在读取数据...
数据读取完成，共 1000000 条记录

=== 任务1：event_type字段统计 ===
event_type
view           250217
purchase       250171
add_to_cart    249879
click          249733
Name: count, dtype: int64

=== 任务2：user_id查询 ===
user_id为-1的记录数：0
user_id以guest-开头的记录数：499687

=== 任务3：device_info解析（操作系统分布） ===
os
Android    250390
iOS        249929
macOS      249851
Windows    249830
Name: count, dtype: int64

分析完成！


In [11]:
import duckdb
import time
import os

# 创建输出目录
os.makedirs("clean_data", exist_ok=True)

# 记录总开始时间
total_start_time = time.time()

# 连接DuckDB（内存模式，支持并行处理）
conn = duckdb.connect(database=':memory:')
# 开启并行查询（提升大数据处理速度）
conn.execute("SET threads TO 8;")

try:
    # ===================== 步骤1：读取原始数据 & 查看字段名和类型 =====================
    print("1. 读取原始CSV数据...")
    read_start = time.time()
    # 注册CSV为虚拟表（直接读取，不加载到Python内存）
    conn.execute("""
        CREATE VIEW large_data AS
        SELECT * FROM 'large_data.csv'
    """)
    
    # 自动获取CSV的字段名和类型（关键：适配字段类型）
    columns = conn.execute("PRAGMA table_info(large_data)").fetchall()
    column_info = {col[1]: col[2] for col in columns}  # 字段名: 类型
    column_names = list(column_info.keys())
    print(f"   ✅ 检测到CSV字段（名: 类型）：{column_info}")
    
    # 获取原始数据总量
    total_raw_rows = conn.execute("SELECT COUNT(*) FROM large_data").fetchone()[0]
    read_time = time.time() - read_start
    print(f"   ✅ 原始数据共 {total_raw_rows:,} 条，读取耗时: {read_time:.2f}秒")

    # ===================== 步骤2：基础数据统计（保留原有统计功能） =====================
    print("\n2. 基础数据分布统计：")
    # 2.1 event_type分布
    print("   - event_type字段分布：")
    event_counts = conn.execute("""
        SELECT event_type, COUNT(*) AS count
        FROM large_data
        GROUP BY event_type
        ORDER BY count DESC
    """).df()
    print(event_counts)

    # 2.2 特殊user_id统计
    user_minus1 = conn.execute("SELECT COUNT(*) FROM large_data WHERE user_id = '-1'").fetchone()[0]
    user_guest = conn.execute("""
        SELECT COUNT(*) FROM large_data
        WHERE user_id = 'guest' OR user_id LIKE 'guest-%'
    """).fetchone()[0]
    print(f"   - user_id为-1的记录数：{user_minus1:,}")
    print(f"   - user_id为guest/guest-开头的记录数：{user_guest:,}")

    # 2.3 设备操作系统分布
    print("   - 设备操作系统分布：")
    os_counts = conn.execute("""
        SELECT 
            TRY_CAST(JSON_EXTRACT(device_info, '$.os') AS VARCHAR) AS os,
            COUNT(*) AS count
        FROM large_data
        WHERE device_info IS NOT NULL AND device_info != '' AND device_info LIKE '%{%'
        GROUP BY os
        HAVING os IS NOT NULL
        ORDER BY count DESC
    """).df()
    print(os_counts)

    # ===================== 步骤3：数据清洗（核心修复：适配event_id类型） =====================
    print("\n3. 开始数据清洗（千万级数据处理）...")
    clean_start = time.time()
    
    # 动态适配字段：优先用action_time替代timestamp
    time_field = "action_time" if "action_time" in column_names else "timestamp"
    ip_field = "ip_address" if "ip_address" in column_names else (
        "ip" if "ip" in column_names else ""
    )
    
    # 核心修复：根据event_id的类型调整过滤条件
    event_id_type = column_info.get("event_id", "").lower()
    if "int" in event_id_type or "bigint" in event_id_type:
        # event_id是整数类型：过滤NULL和非正数（整数无空字符串）
        event_id_filter = "event_id IS NOT NULL AND event_id > 0"
    else:
        # event_id是字符串类型：过滤NULL和空字符串
        event_id_filter = "event_id IS NOT NULL AND event_id != ''"
    
    # 构建清洗SQL（只保留存在的字段）
    clean_sql = f"""
        CREATE TABLE cleaned_data AS
        SELECT
            -- 保留有效event_id（根据类型过滤）
            event_id,
            -- event_type标准化：转为小写，去除首尾空格
            LOWER(TRIM(event_type)) AS event_type,
            -- user_id脱敏：SHA256哈希生成masked_user_id，删除原始user_id
            SHA256(COALESCE(user_id, '')) AS masked_user_id,
            -- 解析并保留device_info（原样保留）
            device_info,
            -- 时间字段（适配实际字段名）
            {time_field}
    """
    # 只有ip字段存在时才添加
    if ip_field:
        clean_sql += f",\n            {ip_field}"
    
    # 补充metadata字段（你的CSV包含该字段，添加进来）
    if "metadata" in column_names:
        clean_sql += f",\n            metadata"
    
    # 补充FROM和WHERE子句（使用适配类型的过滤条件）
    clean_sql += f"""
        FROM large_data
        -- 核心过滤条件：适配event_id类型
        WHERE {event_id_filter}
    """
    
    # 执行清洗SQL
    conn.execute(clean_sql)
    
    # 获取清洗后的数据量
    total_clean_rows = conn.execute("SELECT COUNT(*) FROM cleaned_data").fetchone()[0]
    clean_time = time.time() - clean_start
    print(f"   ✅ 清洗完成，过滤后剩余 {total_clean_rows:,} 条记录，清洗耗时: {clean_time:.2f}秒")

    # ===================== 步骤4：输出Parquet文件（分区存储） =====================
    print("\n4. 写入Parquet文件到clean_data目录...")
    write_start = time.time()
    # 按event_type分区写入（提升后续查询效率）
    conn.execute("""
        COPY cleaned_data
        TO 'clean_data/' 
        (FORMAT PARQUET, PARTITION_BY (event_type), OVERWRITE_OR_IGNORE 1)
    """)
    write_time = time.time() - write_start
    print(f"   ✅ Parquet文件写入完成，耗时: {write_time:.2f}秒")

    # ===================== 步骤5：最终统计 =====================
    total_time = time.time() - total_start_time
    print("\n=== 数据清洗&分析全流程完成 ===")
    print(f"📊 核心统计：")
    print(f"   - 原始数据量：{total_raw_rows:,} 条")
    print(f"   - 清洗后数据量：{total_clean_rows:,} 条")
    print(f"   - 数据过滤率：{(1 - total_clean_rows/total_raw_rows)*100:.2f}%")
    print(f"⏱️  耗时统计：")
    print(f"   - 数据读取：{read_time:.2f}秒")
    print(f"   - 数据清洗：{clean_time:.2f}秒")
    print(f"   - 文件写入：{write_time:.2f}秒")
    print(f"   - 总耗时：{total_time:.2f}秒")

except Exception as e:
    print(f"\n❌ 执行出错：{type(e).__name__}: {str(e)}")
    raise
finally:
    # 确保连接关闭
    conn.close()
    clean_data_abs = os.path.abspath("clean_data")
    print(f"   📍 Parquet文件实际生成路径：{clean_data_abs}")
    print("\n🔌 数据库连接已关闭，流程结束")

1. 读取原始CSV数据...
   ✅ 检测到CSV字段（名: 类型）：{'event_id': 'BIGINT', 'user_id': 'VARCHAR', 'action_time': 'VARCHAR', 'event_type': 'VARCHAR', 'device_info': 'VARCHAR', 'metadata': 'VARCHAR'}
   ✅ 原始数据共 1,000,000 条，读取耗时: 0.34秒

2. 基础数据分布统计：
   - event_type字段分布：
    event_type   count
0         view  250217
1     purchase  250171
2  add_to_cart  249879
3        click  249733
   - user_id为-1的记录数：0
   - user_id为guest/guest-开头的记录数：499,687
   - 设备操作系统分布：
          os   count
0  "Android"  250390
1      "iOS"  249929
2    "macOS"  249851
3  "Windows"  249830

3. 开始数据清洗（千万级数据处理）...
   ✅ 清洗完成，过滤后剩余 1,000,000 条记录，清洗耗时: 0.56秒

4. 写入Parquet文件到clean_data目录...
   ✅ Parquet文件写入完成，耗时: 0.38秒

=== 数据清洗&分析全流程完成 ===
📊 核心统计：
   - 原始数据量：1,000,000 条
   - 清洗后数据量：1,000,000 条
   - 数据过滤率：0.00%
⏱️  耗时统计：
   - 数据读取：0.34秒
   - 数据清洗：0.56秒
   - 文件写入：0.38秒
   - 总耗时：2.03秒
   📍 Parquet文件实际生成路径：g:\Users\caoruijie\vscode.cpp\clean_data

🔌 数据库连接已关闭，流程结束


In [15]:
import duckdb
import os
import pandas as pd

# 精准匹配实际路径
clean_data_path = "g:/Users/caoruijie/vscode.cpp/clean_data"

# 连接DuckDB
conn = duckdb.connect(database=':memory:')

try:
    if not os.path.exists(clean_data_path):
        print(f"❌ 路径不存在：{clean_data_path}")
    else:
        print(f"✅ 读取Parquet文件：{clean_data_path}\n")
        
        # 读取前10行数据并指定字段顺序（核心修改：LIMIT 10）
        df = conn.execute(f"""
            SELECT 
                event_id,
                event_type,
                masked_user_id,
                action_time,
                device_info,
                metadata
            FROM read_parquet('{clean_data_path}/**/*.parquet') 
            LIMIT 10
        """).df()

        # 关键配置：字段内容显示100字符，无截断/无感叹号
        pd.set_option('display.max_columns', None)        # 显示所有列
        pd.set_option('display.width', 3000)              # 超宽显示（适配100字符字段）
        pd.set_option('display.max_colwidth', 100)        # 字段内容显示100个字符
        pd.set_option('display.expand_frame_repr', False) # 禁止列折叠（无感叹号）
        pd.set_option('display.unicode.ambiguous_as_wide', True)
        pd.set_option('display.unicode.east_asian_width', True)

        # 输出前10行数据（字段内容显示100字符）
        print("📊 清洗后数据前10行（字段内容显示100字符）：")
        print(df)
        
        # 先查询总数据量，避免f-string嵌套
        total_count = conn.execute(f"SELECT COUNT(*) FROM read_parquet('{clean_data_path}/**/*.parquet')").fetchone()[0]
        
        # 详细验证（实验报告专用）
        print("\n" + "="*120)
        print("🔍 数据清洗效果验证（前10行样本）：")
        print(f"1. event_type标准化：所有10行均为小写 → {df['event_type'].str.islower().all()}")
        print(f"2. masked_user_id脱敏：10行哈希长度均为64位 → {all(df['masked_user_id'].str.len() == 64)}")
        print(f"3. event_id有效性：10行无空值 → {df['event_id'].isna().sum() == 0}")
        print(f"4. 字段完整性：{df.columns.tolist()} → 全部保留")
        print(f"5. 数据分布：10行中event_type类型 → \n{df['event_type'].value_counts()}")
        print(f"6. 总数据量验证：清洗后共 {total_count:,} 条")

except Exception as e:
    print(f"\n❌ 错误详情：{str(e)}")
finally:
    conn.close()

✅ 读取Parquet文件：g:/Users/caoruijie/vscode.cpp/clean_data

📊 清洗后数据前10行（字段内容显示100字符）：
     event_id   event_type                                                    masked_user_id                 action_time                                                                                          device_info                              metadata
0  4590099789  add_to_cart  95309b92f43e891063c1056be321f8fc7dbca19217a33168288c85c1ab16b98e  2023-09-14T14:08:53.390579  {"device_type": "mobile", "os": "Android", "browser": "Edge", "screen_resolution": "1440x900", "...                        user_segment_A
1  3202415134  add_to_cart  4b3ed65afacad1463d384591ff0224d707d0bbcb855cfb18e6a2233a16aa6b15                  1631186598  {"device_type": "tablet", "os": "Android", "browser": "Chrome", "screen_resolution": "1080x1920"...  http://example.com/page_926?param=59
2  1889560339  add_to_cart  1e112bbd53b7bfe8759b85a764669cd3f2dfee68f7c365a59beffdd4d46d527b  2023-06-05T10:25:33.457538  {"device_type": 